# 🎬 ROTBOTS — Video Plan

---

Configure your video before building it. This plan controls everything downstream.

> **🤔** You're about to design a content machine. What decisions are you making vs. the machine?

---

In [ ]:
# SETUP
import json, re
from pathlib import Path
from IPython.display import display, Markdown
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Setup')

---
## 📐 Video Plan

Set the total length, content mix, and features for your video.

In [ ]:
# ============================================================
# VIDEO PLAN — Configure your video here
# ============================================================

TOPIC = 'The history and ethics of AI-generated art'

SOURCES = [
    'https://en.wikipedia.org/wiki/Artificial_intelligence_art',
]

# Total video length
TOTAL_VIDEO_LENGTH = 60    # seconds (including credits)
SECONDS_PER_SCENE = 5      # each scene/clip length

# Content mix (must add up to <= 1.0, rest = AI-generated)
ARCHIVE_RATIO = 0.0        # 0.0-1.0 — archive.org footage
UPLOAD_RATIO = 0.0          # 0.0-1.0 — your own uploaded footage

# Optional features
ENABLE_CREDITS = True
ENABLE_SUBTITLES = False
ENABLE_MUSIC = False
ENABLE_EFFECTS = True

SESSION_NAME = ''  # Leave empty to auto-generate

In [ ]:
# CALCULATE PLAN
AI_RATIO = 1.0 - ARCHIVE_RATIO - UPLOAD_RATIO
if AI_RATIO < 0: print('❌ Ratios exceed 1.0!'); AI_RATIO=1.0; ARCHIVE_RATIO=0; UPLOAD_RATIO=0
CREDITS_LENGTH = 8 if ENABLE_CREDITS else 0
CONTENT_LENGTH = TOTAL_VIDEO_LENGTH - CREDITS_LENGTH
AI_LENGTH = CONTENT_LENGTH * AI_RATIO
ARCHIVE_LENGTH = CONTENT_LENGTH * ARCHIVE_RATIO
UPLOAD_LENGTH = CONTENT_LENGTH * UPLOAD_RATIO
NUM_AI_SCENES = max(1, int(AI_LENGTH / SECONDS_PER_SCENE))
NUM_ARCHIVE_SCENES = int(ARCHIVE_LENGTH / SECONDS_PER_SCENE)
NUM_UPLOAD_SCENES = int(UPLOAD_LENGTH / SECONDS_PER_SCENE)
TOTAL_CONTENT_SCENES = NUM_AI_SCENES + NUM_ARCHIVE_SCENES + NUM_UPLOAD_SCENES
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5
NARRATION_LENGTH = CONTENT_LENGTH

# Interleaved scene order (Option B: evenly distributed)
scene_order = []; ai_idx=0; arch_idx=0; up_idx=0
for i in range(TOTAL_CONTENT_SCENES):
    placed = False
    if NUM_ARCHIVE_SCENES > 0 and arch_idx < NUM_ARCHIVE_SCENES:
        iv = TOTAL_CONTENT_SCENES / NUM_ARCHIVE_SCENES
        if i >= (arch_idx+1)*iv - iv/2 and i < (arch_idx+1)*iv + iv/2:
            scene_order.append({'type':'archive','index':arch_idx}); arch_idx+=1; placed=True
    if not placed and NUM_UPLOAD_SCENES > 0 and up_idx < NUM_UPLOAD_SCENES:
        iv = TOTAL_CONTENT_SCENES / NUM_UPLOAD_SCENES
        if i >= (up_idx+1)*iv - iv/2 and i < (up_idx+1)*iv + iv/2:
            scene_order.append({'type':'upload','index':up_idx}); up_idx+=1; placed=True
    if not placed:
        if ai_idx < NUM_AI_SCENES: scene_order.append({'type':'ai_generated','index':ai_idx}); ai_idx+=1
        else: scene_order.append({'type':'ai_generated','index':NUM_AI_SCENES-1})

if not SESSION_NAME:
    words = re.sub(r'[^a-zA-Z0-9\s]','',TOPIC.lower()).split()
    SESSION_NAME = '-'.join(words[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
for d in ['','videos','audio','archive_clips','uploads']: (SESSION_DIR/d).mkdir(parents=True,exist_ok=True)

plan = {'topic':TOPIC,'sources':SOURCES,'session_name':SESSION_NAME,'total_video_length':TOTAL_VIDEO_LENGTH,'seconds_per_scene':SECONDS_PER_SCENE,'credits_length':CREDITS_LENGTH,'content_length':CONTENT_LENGTH,'ai_ratio':AI_RATIO,'archive_ratio':ARCHIVE_RATIO,'upload_ratio':UPLOAD_RATIO,'ai_length':AI_LENGTH,'archive_length':ARCHIVE_LENGTH,'upload_length':UPLOAD_LENGTH,'num_ai_scenes':NUM_AI_SCENES,'num_archive_scenes':NUM_ARCHIVE_SCENES,'num_upload_scenes':NUM_UPLOAD_SCENES,'total_content_scenes':TOTAL_CONTENT_SCENES,'words_per_scene':WORDS_PER_SCENE,'narration_length':NARRATION_LENGTH,'scene_order':scene_order,'enable_credits':ENABLE_CREDITS,'enable_subtitles':ENABLE_SUBTITLES,'enable_music':ENABLE_MUSIC,'enable_effects':ENABLE_EFFECTS}
with open(SESSION_DIR/'video_plan.json','w') as f: json.dump(plan,f,indent=2)

print(f'📂 Session: {SESSION_NAME}')
print(f'📐 Video Plan:')
print(f'   Total: {TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s content + {CREDITS_LENGTH}s credits')
print(f'   🤖 AI: {AI_RATIO:.0%} = {AI_LENGTH:.0f}s ({NUM_AI_SCENES} scenes)')
if ARCHIVE_RATIO>0: print(f'   🏛️ Archive: {ARCHIVE_RATIO:.0%} = {ARCHIVE_LENGTH:.0f}s ({NUM_ARCHIVE_SCENES} scenes)')
if UPLOAD_RATIO>0: print(f'   📂 Upload: {UPLOAD_RATIO:.0%} = {UPLOAD_LENGTH:.0f}s ({NUM_UPLOAD_SCENES} scenes)')
print(f'   📝 Narration: {NARRATION_LENGTH:.0f}s ({int(NARRATION_LENGTH*2.5)} words)')
print(f'\n🎬 Scene order:')
for i,s in enumerate(scene_order):
    icon={'ai_generated':'🤖','archive':'🏛️','upload':'📂'}[s['type']]
    print(f'   {i+1}. {icon} {s["type"]} #{s["index"]+1}')
if ENABLE_CREDITS: print(f'   {len(scene_order)+1}. 📜 credits')
print(f'\n✅ Plan saved')

---
*ROTBOTS Workshop — LI-MA 2026*